In [22]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq                          # replaces ChatOpenAI
import os

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [23]:
# from langchain_community.tools import DuckDuckGoSearchRun

# search = DuckDuckGoSearchRun()

# search.invoke("what is the capital of france?")

In [24]:
from langchain.tools import tool

In [25]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response

In [26]:
tool_duckduckgo_search.invoke("What is the capital of France?")

'... of these cities is the capital, keep reading to find out. ... What is the capital city of France? The capital, and largest, city of France is Paris. Besides Paris, what is the capital of France? ... You use it between your head and your toes, the more it works the thinner it grows. What is the Capital of France? Paris ... Paris, the capital city of France, is one of the most famous and influential cities in the world. What is the Capital of France? ... As the capital city of France, the city plays host to the national government of France. However, Paris only became the official capital of France during the reign of Clovis I, in the late 5th and early 6th century.'

In [27]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about persons, places, etc."""

    import wikipedia  # ← add this
    wikipedia.set_user_agent("MyLangChainApp/1.0 (your-email@example.com)")  # ← and this

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia_tool.invoke(query)

    return response

In [28]:
tool_wikipedia_search.invoke("Barak Obama")

"Page: Barack Obama\nSummary: Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.\nBorn in Honolulu, Obama graduated from Columbia University in 1983 with a Bachelor of Arts degree in political science and later worked as a community organizer in Chicago. In 1988, Obama enrolled in Harvard Law School, where he was the first Black president of the Harvard Law Review. He became a civil rights attorney and an academic, teaching constitutional law at the University of Chicago Law School from 1992 to 2004. In 1996, Obama was elected to represent the 13th district in the Illinois Senate, a position he held until 2004, when he successfully ran for the U.S. Senate. In the 2008 presidential ele

In [32]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [33]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [34]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

## Bind Tools

In [36]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info
]

In [39]:
llm_bind = llm.bind_tools(toolkit)

In [40]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rtb021rmn', 'function': {'arguments': '{"name":"John Doe"}', 'name': 'tool_personal_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 553, 'total_tokens': 570, 'completion_time': 0.065434308, 'completion_tokens_details': None, 'prompt_time': 0.043066128, 'prompt_tokens_details': None, 'queue_time': 0.047686531, 'total_time': 0.108500436}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1bb8-5436-7f31-a446-fb5d1d84a369-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'rtb021rmn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 553, 'output_tokens': 17, 'total_tokens': 570})